In [29]:
import json
import pandas as pd
import numpy as np

In [30]:
weights = json.load(open('../data/origin_dest_weights.json', 'r'))
pairs = json.load(open('../data/origin_pairs.json', 'r'))
tract_dem = json.load(open('../data/tract_demographics.json', 'r'))
income_data = pd.read_csv('../data/income.csv')

In [31]:
results = json.load(open('../results/paths/budget/commutes-0-1.json', 'r')) #| json.load(open('./computation_results/weighted/final-2-2.json', 'r'))

In [40]:
df = pd.DataFrame(index=list(results.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for origin_tract, dest_tracts in results.items():
    
    if dest_tracts == {}:
        df.drop(origin_tract, axis=0, inplace=True)
        continue

    total_weights = []
    if not origin_tract in tract_dem:
        df.drop(origin_tract, axis=0, inplace=True)
        continue
    
    majority = tract_dem[origin_tract]
    
    for d, v in dest_tracts.items():
        if not v: continue
        total_weights.append(weights[origin_tract][d])
        
    total_weights = np.array(total_weights)/np.sum(total_weights)
    budgets = []
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for d, v in dest_tracts.items():
        if not v: continue
        budgets = list(v.keys())
        highest, lowest = (budgets[0], budgets[-1])

        n_cameras.append(int(highest))

        unrestricted_time_to_reach.append(v[highest])
        restricted_time_to_reach.append(v[lowest])

    
    w_uttr = unrestricted_time_to_reach @ total_weights
    w_rttr = restricted_time_to_reach @ total_weights
    
    df.loc[df.index == origin_tract] = [
        majority,
        w_uttr,
        w_rttr,
        (w_rttr - w_uttr)/w_uttr,
        w_rttr - w_uttr,
        total_weights @ n_cameras
    ]

df.relative_increase = df.relative_increase.map(lambda x: 0 if pd.isna(x) else x)
df = df.reset_index()
income_data = income_data.loc[income_data.Year == 2022]
income_data.Geography = income_data.Geography.apply(lambda x: x.split(',')[0])
df = pd.merge(df, income_data, right_on="Geography", left_on='index')

new_cols = [d.replace(" ", "_").lower() for d in df.columns]
cols = df.columns.to_list()

c_map = {cols[i]: new_cols[i] for i in range(len(new_cols))}

df = df.rename(columns=c_map)

df.to_csv('../analysis/weighted.csv')
df

,index,majority,shortest_path,shortest_restricted_path,relative_increase,absolute_increase,n_cameras,id_year,year,id_race,race,household_income_by_race,household_income_by_race_moe,geography,id_geography
0,Census Tract 8424,black,903.151653,903.151653,0.000000,0.0,0.0,2022,2022,0,Total,72616,35239.0,Census Tract 8424,14000US17031842400
1,Census Tract 8403,white,712.287909,712.581225,0.000412,0.293316,1.0,2022,2022,0,Total,65909,16606.0,Census Tract 8403,14000US17031840300
2,Census Tract 8411,white,338.29181,338.316804,0.000074,0.024994,0.0587,2022,2022,0,Total,48542,24522.0,Census Tract 8411,14000US17031841100
3,Census Tract 8412,white,501.290984,504.149074,0.005701,2.85809,0.88573,2022,2022,0,Total,60498,6445.0,Census Tract 8412,14000US17031841200
4,Census Tract 8390,white,398.398831,444.274574,0.115150,45.875743,1.010578,2022,2022,0,Total,110306,22152.0,Census Tract 8390,14000US17031839000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629,Census Tract 704,white,416.118236,431.608012,0.037224,15.489776,1.115823,2022,2022,0,Total,243795,31450.0,Census Tract 704,14000US17031070400
630,Census Tract 705,white,426.75833,426.75833,0.000000,0.0,0.0,2022,2022,0,Total,186757,25472.0,Census Tract 705,14000US17031070500
631,Census Tract 1303,white,826.394927,860.944234,0.041807,34.549307,3.440189,2022,2022,0,Total,66154,9707.0,Census Tract 1303,14000US17031130300
632,Census Tract 2922,white,271.476721,272.045533,0.002095,0.568813,0.135988,2022,2022,0,Total,33736,8677.0,Census Tract 2922,14000US17031292200


In [37]:
df = pd.DataFrame(index=list(results.keys()), columns=['majority', 'shortest_path', 'shortest_restricted_path', 'relative_increase', 'absolute_increase', 'n_cameras'])

for origin_tract, dest_tracts in results.items():
    
    if dest_tracts == {}:
        df.drop(origin_tract, axis=0, inplace=True)
        continue

    total_weights = []
    if not origin_tract in tract_dem:
        df.drop(origin_tract, axis=0, inplace=True)
        continue
    
    majority = tract_dem[origin_tract]
    
    for d, v in dest_tracts.items():
        if not v: continue
        total_weights.append(weights[origin_tract][d])
        
    total_weights = np.array(total_weights)/np.sum(total_weights)
    #print(total_weights)
    budgets = []
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for d, v in dest_tracts.items():
        if not v: continue
        budgets = list(v.keys())
        highest, lowest = (budgets[0], budgets[-1])

        n_cameras.append(int(highest))

        unrestricted_time_to_reach.append(v[highest])
        restricted_time_to_reach.append(v[lowest])

    
    w_uttr = unrestricted_time_to_reach @ total_weights
    w_rttr = restricted_time_to_reach @ total_weights
    if total_weights @ n_cameras == 0: df.drop(origin_tract, axis=0, inplace=True)
    df.loc[df.index == origin_tract] = [
        majority,
        w_uttr,
        w_rttr,
        (w_rttr - w_uttr)/w_uttr,
        w_rttr - w_uttr,
        total_weights @ n_cameras
    ]

df.relative_increase = df.relative_increase.map(lambda x: 0 if pd.isna(x) else x)
df.to_csv('./analysis/nonzero-weighted.csv')

OSError: Cannot save file into a non-existent directory: 'analysis'